# 1. Configuración del entorno y carga de datos

Este notebook ha de ser ejecutado después de ejecutar el archivo de R `TCGA_data_extraction.R` que se encuentra en la carpeta `\src`

## 1.1 Importación de Librerías
Importamos las librerías esenciales para la manipulación de datos, el cálculo numérico y la visualización.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuramos el estilo de las visualizaciones para que sean más atractivas
sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

## 1.2. Definición de Rutas
Definimos las rutas a los datos de entrada y a las carpetas de salida utilizando rutas relativas para garantizar la reproducibilidad del proyecto.

In [ ]:
DATA_INPUT_PATH = '../data/raw/'
DATA_PROCESSED_PATH = '../data/processed/'
FIGURES_PATH = '../outputs/figures/'

#Ficheros descargados desde R
COUNTS_FILENAME = 'TCGA-LUAD_star_counts.csv'
CLINICAL_FILENAME = 'TCGA-LUAD_clinical_data_clean.csv'

# Creamos las carpetas de salida si no existen
os.makedirs(DATA_PROCESSED_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)

print(f"Ruta de datos de entrada: {os.path.abspath(DATA_INPUT_PATH)}")
print(f"Ruta de datos procesados: {os.path.abspath(DATA_PROCESSED_PATH)}")

## 1.3. Carga de los Datasets de TCGA

Cargamos la matriz de conteos de expresión y la tabla de datos clínicos en DataFrames de pandas. La primera columna de los CSVs se utilizará como índice.

In [ ]:
print("Cargando la matriz de conteos...")
counts_df = pd.read_csv(
    os.path.join(DATA_INPUT_PATH, COUNTS_FILENAME),
    index_col=0
)

print("Cargando los datos clínicos...")
clinical_df = pd.read_csv(
    os.path.join(DATA_INPUT_PATH, CLINICAL_FILENAME),
    index_col=0
)

print("Datos cargados exitosamente.")

## 1.4. Inspección Inicial

Realizamos una primera verificación de los datos cargados para entender su estructura, dimensiones y tipos de datos.

In [ ]:
print("\n--- Inspección de la Matriz de Conteos ---")
print(f"Dimensiones: {counts_df.shape[0]} muestras, {counts_df.shape[1]} genes")
print("Primeras 5 filas:")
display(counts_df.head())

In [ ]:
print("\n--- Inspección de los Datos Clínicos ---")
print(f"Dimensiones: {clinical_df.shape[0]} muestras, {clinical_df.shape[1]} variables clínicas")
print("\nInformación de las columnas y tipos de datos:")
clinical_df.info()
print("\nPrimeras 5 filas:")
display(clinical_df.head())

In [ ]:
try:
    assert counts_df.shape[0] == clinical_df.shape[0], "El número de muestras NO coincide entre conteos y datos clínicos"
    print("\n[OK] El número de muestras es consistente entre ambos ficheros.")
except AssertionError as e:
    print(f"\n[ERROR] {e}")

# 2. Limpieza y Fusión de Datos

## 2.1. Análisis de los Tipos de Muestra

No todas las 600 muestras corresponden a tejido tumoral primario. Algunas pueden ser tejido normal adyacente o metástasis. Usamos la columna `definition` para identificar y seleccionar solo las muestras que nos interesan para la deconvolución.

In [ ]:
print("--- Análisis de los Tipos de Muestra ---")
sample_types = clinical_df['definition'].value_counts()
print("Distribución de los tipos de muestra en la cohorte:")
print(sample_types)

In [ ]:
# Visualizamos la distribución
plt.figure(figsize=(8, 6))
sns.countplot(y=clinical_df['definition'])
plt.title('Distribución de Tipos de Muestra en la cohorte TCGA-LUAD')
plt.xlabel('Número de Muestras')
plt.ylabel('Tipo de Muestra')
plt.show()

## 2.2. Filtrado de Muestras Tumorales Primarias
Nos quedamos exclusivamente con las muestras anotadas como "Primary solid Tumor". Este será nuestro conjunto de datos principal para el análisis.

In [ ]:
print("\n--- Filtrando por Tumores Sólidos Primarios ---")
primary_tumor_barcodes = clinical_df[clinical_df['definition'] == 'Primary solid Tumor'].index

print(f"Se han identificado {len(primary_tumor_barcodes)} muestras de tumores primarios.")

In [ ]:
common_barcodes = list(set(primary_tumor_barcodes) & set(counts_df.index))
print(f"De estas, {len(common_barcodes)} también están presentes en la matriz de conteos.")

## 2.3. Sincronización Final de los DataFrames

Aplicamos el filtro a ambos DataFrames para asegurar que contienen exactamente el mismo conjunto de muestras y en el mismo orden.


In [ ]:
# Filtramos ambos dataframes
counts_df_tumor = counts_df.loc[common_barcodes]
clinical_df_tumor = clinical_df.loc[common_barcodes]

In [ ]:
# Verificación final de las dimensiones
print("\n--- Dimensiones después del filtrado ---")
print(f"Matriz de conteos: {counts_df_tumor.shape[0]} muestras, {counts_df_tumor.shape[1]} genes")
print(f"Datos clínicos: {clinical_df_tumor.shape[0]} muestras, {clinical_df_tumor.shape[1]} variables")

In [ ]:
# Aserción final para garantizar la coherencia
try:
    assert counts_df_tumor.shape[0] == clinical_df_tumor.shape[0]
    assert all(counts_df_tumor.index == clinical_df_tumor.index)
    print("\n[OK] Los DataFrames de tumores están perfectamente sincronizados.")
except AssertionError:
    print("\n[ERROR] ¡Los DataFrames no están sincronizados después del filtrado!")

In [ ]:
# Mostramos un resumen de los datos limpios
print("\nPrimeras filas de los datos clínicos de los tumores:")
display(clinical_df_tumor.head())

# 3. Control de Calidad (QC) de los Datos de Expresión

## 3.1. Cálculo de Métricas de Calidad

Antes de la deconvolución, evaluamos la calidad técnica de cada muestra tumoral.
Calculamos dos métricas clave:
1. `total_counts`: La profundidad de secuenciación.
2. `n_genes_detected`: La complejidad de la librería.

Estas métricas se añadirán como nuevas columnas a nuestra tabla de datos clínicos.

In [ ]:
print("--- Calculando métricas de QC para las muestras tumorales ---")

# 1. Calcular el total de conteos por muestra (suma a lo largo de las columnas de genes)
clinical_df_tumor['total_counts'] = counts_df_tumor.sum(axis=1)

# 2. Calcular el número de genes detectados por muestra (genes con conteo > 0)
clinical_df_tumor['n_genes_detected'] = (counts_df_tumor > 0).sum(axis=1)

print("Métricas de QC calculadas y añadidas a la tabla clínica.")
display(clinical_df_tumor[['total_counts', 'n_genes_detected']].describe())

## 3.2. Visualización de las Métricas de Calidad

Visualizamos la distribución de estas métricas para identificar posibles muestras atípicas (outliers) que podrían necesitar ser excluidas del análisis.

In [ ]:
print("\n--- Visualizando distribuciones de las métricas de QC ---")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histograma y boxplot para total_counts
sns.histplot(clinical_df_tumor['total_counts'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribución de Conteos Totales por Muestra')
axes[0].set_xlabel('Conteos Totales (Profundidad de Secuenciación)')

sns.boxplot(y=clinical_df_tumor['total_counts'], ax=axes[1])
axes[1].set_title('Boxplot de Conteos Totales')
axes[1].set_ylabel('Conteos Totales')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Histograma y boxplot para n_genes_detected
sns.histplot(clinical_df_tumor['n_genes_detected'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribución de Genes Detectados por Muestra')
axes[0].set_xlabel('Número de Genes Detectados (Complejidad)')

sns.boxplot(y=clinical_df_tumor['n_genes_detected'], ax=axes[1])
axes[1].set_title('Boxplot de Genes Detectados')
axes[1].set_ylabel('Número de Genes Detectados')

plt.tight_layout()
plt.show()

## 3.3. Decisión sobre el Filtrado por Calidad

En la gráfica de la distribución de conteos se observa que es bimodal, con dos picos diferenciados en aprox 0.4 y 0.7 conteos totales. Esto podría indicar un *efecto de lote* ( ya sea porque provienen de protocolos diferentes o porque hay algún parámetro clínico que podría estar afectando). En la gráfica de Genes detectados se observa una buena proporción de genes con unos 32000-36000 genes detectados y una ligera simetría de la distribución con una cola hacia la derecha, lo que indica que hay muestras con mayor cantidad de genes detectados. Por lo tanto, se decide no hacer filtrado según la calidad de las muestras.

In [ ]:
print("\nAnálisis de QC completado. No se han identificado outliers de baja calidad que requieran filtrado.")
print("Se procede con las 539 muestras tumorales.")

# Creamos las variables finales como copias de las de entrada
counts_df_final = counts_df_tumor.copy()
clinical_df_final = clinical_df_tumor.copy()

# 4. Caracterización de la Cohorte Clínica

## 4.1. Resumen de la Cohorte Final
Con nuestras 539 muestras tumorales de alta calidad, procedemos a describir las características clínicas y demográficas de los pacientes.

In [ ]:
print("--- Caracterizando la cohorte final ---")
print(f"Número total de pacientes/muestras en el análisis: {clinical_df_final.shape[0]}")
display(clinical_df_final.describe(include='all'))

## 4.2. Análisis de Variables Categóricas
Visualizamos la distribución de las variables categóricas más importantes para entender la composición de nuestra cohorte.

In [ ]:
print("\n--- Visualizando Variables Categóricas ---")

# Creamos una figura con varios subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Distribución de Variables Clínicas Categóricas', fontsize=16)

# Gráfico 1: Estadio del Tumor (AJCC Pathologic Stage)
sns.countplot(y=clinical_df_final['ajcc_pathologic_stage'].sort_values(), ax=axes[0, 0])
axes[0, 0].set_title('Distribución por Estadio del Tumor')
axes[0, 0].set_xlabel('Número de Pacientes')
axes[0, 0].set_ylabel('Estadio Patológico AJCC')

# Gráfico 2: Sexo (Gender)
sns.countplot(x=clinical_df_final['gender'], ax=axes[0, 1])
axes[0, 1].set_title('Distribución por Sexo')
axes[0, 1].set_xlabel('Sexo')
axes[0, 1].set_ylabel('Número de Pacientes')

# Gráfico 3: Estado Vital (Vital Status)
sns.countplot(x=clinical_df_final['vital_status'], ax=axes[1, 0])
axes[1, 0].set_title('Distribución por Estado Vital')
axes[1, 0].set_xlabel('Estado Vital')
axes[1, 0].set_ylabel('Número de Pacientes')

# Gráfico 4: Raza (Race)
sns.countplot(y=clinical_df_final['race'], ax=axes[1, 1])
axes[1, 1].set_title('Distribución por Raza')
axes[1, 1].set_xlabel('Número de Pacientes')
axes[1, 1].set_ylabel('Raza')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 4.3. Análisis de Variables Numéricas
Analizamos la distribución de la edad de los pacientes.

In [ ]:
print("\n--- Visualizando Variables Numéricas ---")

plt.figure(figsize=(8, 6))
sns.histplot(clinical_df_final['age_at_index'], bins=30, kde=True)
plt.title('Distribución de la Edad en el Momento del Diagnóstico')
plt.xlabel('Edad (años)')
plt.ylabel('Número de Pacientes')
plt.show()

## 4.4. Preparación de Datos para el Análisis de Supervivencia

Para poder realizar análisis de supervivencia más adelante (ej. Kaplan-Meier), creamos dos columnas estandarizadas:
1. `survival_time`: El tiempo total de seguimiento en días.
2. `event_status`: Un indicador binario (1 si el paciente falleció, 0 si sigue vivo).

In [ ]:
print("\n--- Preparando variables para el análisis de supervivencia ---")

# 1. Crear la columna de tiempo de supervivencia
# Con np.where, si vital_status es 'Dead' se usa days_to_death, si no, se usa days_to_follow_up
clinical_df_final['survival_time'] = np.where(
    clinical_df_final['vital_status'] == 'Dead',
    clinical_df_final['days_to_death'],
    clinical_df_final['days_to_last_follow_up']
)

# 2. Crear la columna de evento
# Igual que antes pero 1 para 'Dead', 0 para 'Alive'
clinical_df_final['event_status'] = np.where(
    clinical_df_final['vital_status'] == 'Dead',
    1,
    0
)

# 3. Verificación y limpieza
print(f"Número de valores nulos en 'survival_time' antes de limpiar: {clinical_df_final['survival_time'].isna().sum()}")

# Eliminamos las filas donde el tiempo de supervivencia es nulo o negativo (datos de mala calidad)
initial_samples = len(clinical_df_final)
clinical_df_final.dropna(subset=['survival_time'], inplace=True)
clinical_df_final = clinical_df_final[clinical_df_final['survival_time'] >= 0]

print(f"Se eliminaron {initial_samples - len(clinical_df_final)} muestras debido a datos de supervivencia inválidos.")
print(f"Número final de muestras con datos de supervivencia válidos: {len(clinical_df_final)}")

# Verificamos que los dataframes sigan sincronizados
counts_df_final = counts_df_final.loc[clinical_df_final.index]
assert all(counts_df_final.index == clinical_df_final.index)
print("\n[OK] Los dataframes de conteos y clínicos siguen sincronizados.")

# Mostramos las nuevas columnas
display(clinical_df_final[['vital_status', 'days_to_death', 'days_to_last_follow_up', 'survival_time', 'event_status']].head())

# 5. Análisis Exploratorio de la Expresión Global

## 5.1. Normalización de los Datos de Conteos

Para poder comparar la expresión génica entre muestras, primero debemos normalizar los conteos crudos para corregir las diferencias en la profundidad de secuenciación. Utilizaremos el método de Cuentas Por Millón (CPM) seguido de una transformación logarítmica (log2(CPM + 1)).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("--- Normalizando la matriz de conteos ---")

# Calculamos los factores de escala (total de conteos por muestra / 1 millón)
cpm_factors = counts_df_final.sum(axis=1) / 1e6

# Aplicamos la normalización a CPM
counts_cpm_df = counts_df_final.div(cpm_factors, axis=0)

# Aplicamos la transformación logarítmica (log2(x + 1) para manejar los ceros)
counts_log_cpm_df = np.log2(counts_cpm_df + 1)

print("Normalización completada.")
print("Dimensiones de la matriz normalizada:", counts_log_cpm_df.shape)
display(counts_log_cpm_df.head())

## 5.2. Análisis de Componentes Principales (PCA)

Realizamos un PCA para reducir la dimensionalidad de los datos de expresión y visualizar las principales fuentes de variación entre las muestras. Antes del PCA, es estándar escalar los datos para que cada gen tenga media 0 y varianza 1.

In [ ]:
print("\n--- Realizando PCA sobre los datos normalizados ---")

# 1. Escalar los datos
scaler = StandardScaler()
scaled_data = scaler.fit_transform(counts_log_cpm_df)

# 2. Ejecutar PCA
pca = PCA(n_components=10) # Calculamos los primeros 10 PCs
principal_components = pca.fit_transform(scaled_data)

# 3. Crear un DataFrame con los resultados del PCA
pca_df = pd.DataFrame(
    data=principal_components,
    columns=[f'PC{i+1}' for i in range(10)],
    index=counts_log_cpm_df.index
)

# 4. Unir los PCs con los datos clínicos para facilitar la visualización
pca_clinical_df = pd.concat([pca_df, clinical_df_final], axis=1)

# 5. Analizar la varianza explicada por cada componente
explained_variance = pca.explained_variance_ratio_ * 100
print(f"Varianza explicada por PC1: {explained_variance[0]:.2f}%")
print(f"Varianza explicada por PC2: {explained_variance[1]:.2f}%")

plt.figure(figsize=(8, 6))
plt.bar(range(1, 11), explained_variance)
plt.xlabel('Componente Principal')
plt.ylabel('Porcentaje de Varianza Explicada')
plt.title('Varianza Explicada por los Primeros 10 PCs')
plt.show()

## 5.3. Visualización y Correlación del PCA

Visualizamos los dos primeros componentes principales y coloreamos cada muestra según sus características clínicas para identificar posibles patrones o agrupaciones.

In [ ]:
print("\n--- Visualizando los resultados del PCA ---")

# Creamos una figura con varios subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 16))
fig.suptitle('PCA de Muestras de TCGA-LUAD coloreado por Variables Clínicas', fontsize=20)

# Gráfico 1: Coloreado por Estadio del Tumor
sns.scatterplot(
    data=pca_clinical_df,
    x='PC1', y='PC2',
    hue='ajcc_pathologic_stage',
    alpha=0.7,
    s=50,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Coloreado por Estadio del Tumor')
axes[0, 0].legend(title='Estadio', bbox_to_anchor=(1.05, 1), loc='upper left')


# Gráfico 2: Coloreado por Estado Vital
sns.scatterplot(
    data=pca_clinical_df,
    x='PC1', y='PC2',
    hue='vital_status',
    alpha=0.7,
    s=50,
    ax=axes[0, 1]
)
axes[0, 1].set_title('Coloreado por Estado Vital')
axes[0, 1].legend(title='Estado Vital')

# Gráfico 3: Coloreado por Sexo
sns.scatterplot(
    data=pca_clinical_df,
    x='PC1', y='PC2',
    hue='gender',
    alpha=0.7,
    s=50,
    ax=axes[1, 0]
)
axes[1, 0].set_title('Coloreado por Sexo')
axes[1, 0].legend(title='Sexo')

# Gráfico 4: Coloreado por Profundidad de Secuenciación (Efecto de Lote)
pca_clinical_df['log10_total_counts'] = np.log10(pca_clinical_df['total_counts'])
sns.scatterplot(
    data=pca_clinical_df,
    x='PC1', y='PC2',
    hue='log10_total_counts',
    palette='viridis',
    alpha=0.7,
    s=50,
    ax=axes[1, 1]
)
axes[1, 1].set_title('Coloreado por Profundidad de Secuenciación (log10)')
axes[1, 1].legend(title='log10(Conteos Totales)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 6. Guardado de los Datos Finales

Ahora, guardamos los DataFrames finales:
1. `counts_df_final`: La matriz de conteos crudos (muestras x genes) para la deconvolución.
2. `clinical_df_final`: La tabla de metadatos clínicos, limpia y con las variables de supervivencia.

Utilizaremos el formato Parquet por su eficiencia en velocidad y almacenamiento.

In [ ]:
print("--- Guardando los datos finales para la deconvolución ---")

# Definimos los nombres de los ficheros de salida
COUNTS_OUTPUT_FILENAME = 'TCGA-LUAD_counts_for_deconvolution.parquet'
CLINICAL_OUTPUT_FILENAME = 'TCGA-LUAD_clinical_for_deconvolution.parquet'

# Construimos las rutas completas
counts_output_path = os.path.join(DATA_PROCESSED_PATH, COUNTS_OUTPUT_FILENAME)
clinical_output_path = os.path.join(DATA_PROCESSED_PATH, CLINICAL_OUTPUT_FILENAME)

# Guardamos los DataFrames en formato Parquet
counts_df_final.to_parquet(counts_output_path, engine='pyarrow')
clinical_df_final.to_parquet(clinical_output_path, engine='pyarrow')

print(f"Matriz de conteos guardada en: {counts_output_path}")
print(f"Datos clínicos guardados en: {clinical_output_path}")

# 7. Exportación de Datos para Deconvolución en R

Finalmente, guardamos la matriz de conteos de bulk y la tabla clínica en formato TSV (Tab-Separated Values) para su uso en el pipeline de deconvolución que se ejecutará en R con el paquete `immunedeconv`.


In [ ]:
BULK_COUNTS_FILENAME = 'TCGA-LUAD_counts_for_deconvolution.parquet'
BULK_CLINICAL_FILENAME = 'TCGA-LUAD_clinical_for_deconvolution.parquet'
counts_df_final = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_COUNTS_FILENAME))
clinical_df_final = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_CLINICAL_FILENAME))

In [ ]:
print("--- Exportando datos de bulk a formato TSV para R ---")

# Rutas de salida
BULK_TSV_FILENAME = 'TCGA-LUAD_bulk_for_R.tsv'
CLINICAL_TSV_FILENAME = 'TCGA-LUAD_clinical_for_R.tsv'
bulk_tsv_path = os.path.join(DATA_PROCESSED_PATH, BULK_TSV_FILENAME)
clinical_tsv_path = os.path.join(DATA_PROCESSED_PATH, CLINICAL_TSV_FILENAME)

# Preparar y guardar la matriz de conteos
counts_to_save_r = counts_df_final.copy()
counts_to_save_r.index.name = 'sample_id' #Columna implícita, mejor para R
counts_to_save_r.to_csv(bulk_tsv_path, sep='\t')
print(f"Matriz de conteos de bulk guardada en: {bulk_tsv_path}")

# Preparar y guardar los datos clínicos
clinical_to_save_r = clinical_df_final.copy()
clinical_to_save_r.index.name = 'sample_id'
clinical_to_save_r.to_csv(clinical_tsv_path, sep='\t')
print(f"Datos clínicos guardados en: {clinical_tsv_path}")